# MNIST MLP3: SGD with momentum vs Local-Delta ECS WW-PGD

This notebook runs **five paired baseline runs** and **five paired optimizer-extension runs** for ten epochs on the standard

$$784 \rightarrow 512 \rightarrow 512 \rightarrow 10$$

MNIST MLP. At the end of every epoch, the extension forms the completed optimizer displacement

$$\Delta W = W_{\rm end}-W_{\rm start},$$

computes the self-consistent ECS of the proposed endpoint $W_{\rm end}$, and applies the fractional local-delta correction

$$\Delta W_{\rm new}=\Delta W-\eta\,\Delta W_\perp.$$

The layer is first oriented as $N\times M$ with $N\ge M$, matching the trace-log optimizer and WeightWatcher convention. Consequently, the ECS projection acts on the right for tall/square matrices and on the left after mapping back for originally wide matrices.


## Dependencies

WeightWatcher is required for this experiment. The cell below installs it when necessary rather than silently replacing the requested WeightWatcher measurements with a proxy.


In [ ]:
import importlib
import subprocess
import sys

for import_name, pip_name in {
    "weightwatcher": "weightwatcher>=0.7.7",
}.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
package_root = None
for root in [cwd, *cwd.parents]:
    direct = root / "wwpgd_local_delta"
    nested = root / "optimizers" / "wwpgd_local_delta" / "wwpgd_local_delta"
    if direct.is_dir():
        package_root = root
        break
    if nested.is_dir():
        package_root = nested.parent
        break
if package_root is None:
    raise RuntimeError(
        "Could not find wwpgd_local_delta. Run this notebook from the optimizer folder or repository root."
    )
if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))
print("Using package root:", package_root)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wwpgd_local_delta import MNISTRunConfig
from wwpgd_local_delta.mnist_experiment import (
    run_mnist_comparison,
    summarize_final_performance,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)


def plot_metric(df, metric, title, ylabel):
    if metric not in df.columns:
        print(f"Skipping {metric}: column is absent")
        return
    clean = df.copy()
    clean[metric] = pd.to_numeric(clean[metric], errors="coerce")
    clean = clean.dropna(subset=[metric])
    if clean.empty:
        print(f"Skipping {metric}: no numeric values")
        return
    fig, ax = plt.subplots(figsize=(9, 5))
    for arm, arm_df in clean.groupby("arm"):
        for _, seed_df in arm_df.groupby("seed"):
            seed_df = seed_df.sort_values("epoch")
            ax.plot(seed_df["epoch"], seed_df[metric], alpha=0.22, linewidth=1.0)
        grouped = arm_df.groupby("epoch")[metric].agg(["mean", "std"]).reset_index()
        x = grouped["epoch"].to_numpy()
        mean = grouped["mean"].to_numpy()
        std = grouped["std"].fillna(0.0).to_numpy()
        ax.plot(x, mean, marker="o", linewidth=2.5, label=f"{arm}: mean")
        ax.fill_between(x, mean - std, mean + std, alpha=0.15)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


def plot_layer_metric(df, metric, title, ylabel, layer_column="layer_name"):
    if metric not in df.columns or layer_column not in df.columns:
        print(f"Skipping {metric}: required columns are absent")
        return
    clean = df.copy()
    clean[metric] = pd.to_numeric(clean[metric], errors="coerce")
    clean = clean.dropna(subset=[metric, layer_column])
    if clean.empty:
        print(f"Skipping {metric}: no numeric values")
        return
    for layer, layer_df in clean.groupby(layer_column):
        fig, ax = plt.subplots(figsize=(9, 5))
        for arm, arm_df in layer_df.groupby("arm"):
            for _, seed_df in arm_df.groupby("seed"):
                seed_df = seed_df.sort_values("epoch")
                ax.plot(seed_df["epoch"], seed_df[metric], alpha=0.22, linewidth=1.0)
            grouped = arm_df.groupby("epoch")[metric].agg(["mean", "std"]).reset_index()
            x = grouped["epoch"].to_numpy()
            mean = grouped["mean"].to_numpy()
            std = grouped["std"].fillna(0.0).to_numpy()
            ax.plot(x, mean, marker="o", linewidth=2.5, label=f"{arm}: mean")
            ax.fill_between(x, mean - std, mean + std, alpha=0.15)
        ax.set_title(f"{title}: {layer}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()


## Five paired runs

The baseline and extension arms load the exact same initial state for each seed and receive the same shuffled minibatch order. The correction uses the **epoch-end ECS**, which is the new ECS of the optimizer's proposed endpoint. All matrix-valued MLP layers are instrumented and corrected.


In [ ]:
CONFIG = MNISTRunConfig(
    optimizer_kind="sgd_momentum",
    epochs=10,
    seeds=(1337, 2027, 4099, 7919, 104729),
    correction_fraction=0.25,
    apply_every_epochs=1,
    warmup_epochs=0,
    normalization_gamma=0.0,
    ecs_reference="epoch_end",
    corrected_parameters=None,
    data_dir="./data",
    output_dir="./runs_sgd_momentum_local_delta_ecs",
    ww_enabled=True,
    ww_required=True,
    ww_min_evals=8,
    ww_svd_method="accurate",
)
CONFIG


In [ ]:
result = run_mnist_comparison(CONFIG, progress=True)
result.save(CONFIG.output_dir)
print("Saved outputs to", Path(CONFIG.output_dir).resolve())


## Fail-fast validation

These assertions verify the experimental pairing and the exact local-delta algebra before interpreting any accuracy or spectral curve.


In [ ]:
performance = result.performance.copy()
spectral = result.spectral.copy()
corrections = result.corrections.copy()

assert set(performance["arm"]) == {"baseline", "local_delta_ecs"}
assert performance.groupby("arm")["seed"].nunique().to_dict() == {
    "baseline": 5,
    "local_delta_ecs": 5,
}
assert int(performance["epoch"].min()) == 0
assert int(performance["epoch"].max()) == 10
assert performance.groupby(["seed", "epoch", "arm"]).size().eq(1).all()
assert performance.groupby("seed")["initial_state_checksum"].nunique().eq(1).all()

assert not corrections.empty
ok = corrections[corrections["status"] == "ok"].copy()
assert not ok.empty
assert float(ok["damping_error"].max()) < 1e-4
assert float(ok["pythagorean_error"].max()) < 1e-4
assert float(ok["correction_identity_error"].max()) < 1e-6
assert np.allclose(
    ok["removed_fraction_of_base"].to_numpy(),
    CONFIG.correction_fraction * ok["orthogonal_fraction"].to_numpy(),
    rtol=1e-4,
    atol=1e-7,
)
assert set(ok["reference"]) == {"epoch_end"}
assert set(ok["projection_side"]).issubset({"left", "right"})

assert not spectral.empty
assert spectral["diagnostic_source"].eq("weightwatcher").all(), spectral[
    ["run_label", "epoch", "layer_name", "diagnostic_source", "diagnostic_error"]
].query("diagnostic_source != 'weightwatcher'")

print("All pairing, projection, and WeightWatcher checks passed.")


## Summary tables


In [ ]:
display(summarize_final_performance(result.performance))
display(
    result.performance.groupby(["arm", "epoch"])[
        ["train_acc", "test_acc", "train_loss", "test_loss"]
    ].agg(["mean", "std"]).tail(8)
)
display(result.corrections.tail(30))
display(result.spectral.tail(30))


## Task metrics


In [ ]:
plot_metric(result.performance, "train_acc", "Training accuracy", "Accuracy")
plot_metric(result.performance, "test_acc", "Test accuracy", "Accuracy")
plot_metric(result.performance, "train_loss", "Training cross-entropy", "Loss")
plot_metric(result.performance, "test_loss", "Test cross-entropy", "Loss")
plot_metric(
    result.performance,
    "correction_train_loss_delta",
    "Immediate train-loss change caused by epoch correction",
    "Post minus pre correction loss",
)
plot_metric(
    result.performance,
    "correction_test_loss_delta",
    "Immediate test-loss change caused by epoch correction",
    "Post minus pre correction loss",
)
plot_metric(
    result.performance,
    "correction_test_acc_delta",
    "Immediate test-accuracy change caused by epoch correction",
    "Post minus pre correction accuracy",
)


## WeightWatcher and local spectral metrics


In [ ]:
for metric, title, ylabel in [
    ("alpha", "WeightWatcher alpha", "alpha"),
    ("alpha_weighted", "WeightWatcher weighted alpha", "weighted alpha"),
    ("weighted_alpha", "WeightWatcher weighted alpha", "weighted alpha"),
    ("ERG_gap", "WeightWatcher ERG gap", "ERG gap"),
    ("detX_num", "WeightWatcher detX retained count", "detX_num"),
    ("num_pl_spikes", "WeightWatcher power-law retained count", "num_pl_spikes"),
    ("stable_rank", "WeightWatcher stable rank", "stable rank"),
    ("spectral_norm", "WeightWatcher spectral norm", "spectral norm"),
    ("log_norm", "WeightWatcher log norm", "log norm"),
    ("alpha_proxy", "Direct-SVD rank-slope alpha audit", "alpha proxy"),
    ("rank_slope_q", "Direct-SVD rank slope q", "q"),
    ("ecs_rank_local", "Local self-consistent ECS rank", "rank"),
    ("ecs_fraction_local", "Local ECS rank fraction", "rank / spectral count"),
    ("trace_log_per_eval_local", "Local ECS trace-log residual", "mean trace log"),
    ("bulk_effective_count_local", "Local effective bulk count", "effective count"),
]:
    plot_layer_metric(result.spectral, metric, title, ylabel)


## Local-delta mechanism checks


In [ ]:
correction_plot = result.corrections.rename(columns={"parameter": "layer_name"})
for metric, title, ylabel in [
    ("orthogonal_fraction", "Orthogonal fraction before correction", "||Delta_perp|| / ||Delta||"),
    ("post_orthogonal_fraction", "Orthogonal fraction after correction", "||Delta_perp,new|| / ||Delta||"),
    ("removed_fraction_of_base", "Removed fraction of completed epoch displacement", "||removed|| / ||Delta||"),
    ("observed_orthogonal_damping", "Observed orthogonal damping", "post / pre orthogonal norm"),
    ("damping_error", "Error in requested fractional damping", "absolute error"),
    ("pythagorean_error", "ECS decomposition orthogonality error", "relative error"),
    ("correction_identity_error", "Local-delta algebra identity error", "relative error"),
    ("ecs_rank", "ECS rank used by the correction", "rank"),
    ("ecs_fraction", "ECS rank fraction used by the correction", "rank / spectral count"),
    ("trace_log_per_eval", "ECS trace-log residual used by the correction", "mean trace log"),
]:
    plot_layer_metric(correction_plot, metric, title, ylabel)
